In [2]:
#Import Libraries
import os
import numpy as np
import cv2 as cv
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix


In [3]:
# function to convert imag to pixel array 
import os
def im_to_array(path):
    im_list = []
    for filename in os.listdir(path): # Explain file in each filder
        if filename.endswith(".png"): #read if there is png file
            img =cv.imread(path + "/" + filename, cv.IMREAD_GRAYSCALE) #read the image in gray scale
            im_list.append(img)
    return np.array(im_list).reshape(len(im_list),-1) 

In [4]:
# function to convert images in a folder to pixel array
import os
def im_to_array(path):
    im_list = []
    for filename in os.listdir(path):
        if filename.lower().endswith((".jpg", ".jpeg", ".png")):  # accept all 3
            img = cv.imread(path + "/" + filename, cv.IMREAD_GRAYSCALE)
            if img is None:
                continue                          # skip unreadable files
            img = cv.resize(img, (64, 64))        # resize so all photos match
            im_list.append(img)
    return np.array(im_list).reshape(len(im_list), -1)

In [5]:
# Prepare Dataset
im_1 = im_to_array("E:\\Year3 ITC\\I3 AMS s2\\Intro to ML\\Final_Project\\Cat")
y_1 = np.zeros(im_1.shape[0])                 # Cat -> label 0

im_2 = im_to_array("E:\\Year3 ITC\\I3 AMS s2\\Intro to ML\\Final_Project\\Dog")
y_2 = np.ones(im_2.shape[0])                  # Dog -> label 1

X = np.concatenate((im_1, im_2), axis=0)
y = np.concatenate((y_1, y_2), axis=0)

print("Cat images:", im_1.shape[0])
print("Dog images:", im_2.shape[0])
print("X shape:", X.shape, "  y shape:", y.shape)

Cat images: 100
Dog images: 100
X shape: (200, 4096)   y shape: (200,)


In [6]:
#Save 
np.save("X_train.npy", X)
np.save("y_train.npy", y)

In [7]:
# Scale X and convert array to tensor 

X = X / 255.0
tX = torch.tensor(X, dtype=torch.float32)
ty = torch.tensor(y, dtype=torch.long)
print(tX.shape)   # should print torch.Size([200, 4096])

torch.Size([200, 4096])


In [8]:
# Define  model 
def my_ANN(tX, W1, b1, W2, b2, W3, b3):
    Z1 = torch.matmul(tX, W1) + b1
    A1 = torch.relu(Z1)
    Z2 = torch.matmul(A1, W2) + b2
    A2 = torch.relu(Z2)
    Z3 = torch.matmul(A2, W3) + b3
    
    return Z3

In [9]:
# Training
W1 = torch.randn(4096, 16, requires_grad=True)   # 4096 inputs (64*64), not 784
b1 = torch.randn(1, 16, requires_grad=True)
W2 = torch.randn(16, 8, requires_grad=True)
b2 = torch.randn(1, 8, requires_grad=True)
W3 = torch.randn(8, 2, requires_grad=True)       # 2 classes (cat, dog), not 3
b3 = torch.randn(1, 2, requires_grad=True)

cost_func = nn.CrossEntropyLoss()

optimizer = optim.SGD([W1, b1, W2, b2, W3, b3], lr=0.1)

for epoch in range(1, 1001):
    Z = my_ANN(tX, W1, b1, W2, b2, W3, b3)
    loss = cost_func(Z, ty)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    if epoch % 50 == 0:                          # only print every 50 epochs
        print(f"Epoch: {epoch:03d}, Loss: {loss.item():.4f}")


Epoch: 050, Loss: 0.6931
Epoch: 100, Loss: 0.6931
Epoch: 150, Loss: 0.6931
Epoch: 200, Loss: 0.6931
Epoch: 250, Loss: 0.6931
Epoch: 300, Loss: 0.6931
Epoch: 350, Loss: 0.6931
Epoch: 400, Loss: 0.6931
Epoch: 450, Loss: 0.6931
Epoch: 500, Loss: 0.6931
Epoch: 550, Loss: 0.6931
Epoch: 600, Loss: 0.6931
Epoch: 650, Loss: 0.6931
Epoch: 700, Loss: 0.6931
Epoch: 750, Loss: 0.6931
Epoch: 800, Loss: 0.6931
Epoch: 850, Loss: 0.6931
Epoch: 900, Loss: 0.6931
Epoch: 950, Loss: 0.6931
Epoch: 1000, Loss: 0.6931


In [10]:
# Evaluate 
Z = my_ANN(tX, W1, b1, W2, b2, W3, b3)
y_pred= torch.argmax(Z, 1)
acc=(y_pred == ty).float().mean()
print(f"Accuracy on set: {acc*100: .2f}%")

Accuracy on set:  50.00%


In [11]:
#split dataset
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=142, shuffle=True)

# Display shapes
print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)

print("y_train shape:", y_train.shape)
print("y_test shape :", y_test.shape)

X_train shape: (140, 4096)
X_test shape : (60, 4096)
y_train shape: (140,)
y_test shape : (60,)


In [12]:
#scale X and convert array to tensor
X_train = X_train / 255.0
X_test = X_test / 255.0

X_train = X_train.reshape(X_train.shape[0], -1)
X_test = X_test.reshape(X_test.shape[0], -1)

tX_train = torch.tensor(X_train, dtype=torch.float32)
tx_test = torch.tensor(X_test, dtype=torch.float32)

ty_train = torch.tensor(y_train, dtype=torch.long)
ty_test = torch.tensor(y_test, dtype=torch.long)

print(tX_train.shape)
print(tx_test.shape)

torch.Size([140, 4096])
torch.Size([60, 4096])


In [13]:
#Define Model
def relu(X):
    return torch.maximum(X, torch.tensor(0.0))
def my_ANN(X, w1, b1, w2, b2, w3, b3):
    Z1 = X @ w1 + b1
    A1 = relu(Z1)

    Z2 = A1 @ w2 + b2
    A2 = relu(Z2)

    Z3 = A2 @ w3 + b3

    return Z3

In [14]:
#evaluation

Z = my_ANN(tX_train, w1, b1, w2, b2, w3, b3)
y_pred = torch.argmax(Z, dim=1)
accuracy = (y_pred == ty_train).float().mean()
print(f"Accuracy: {accuracy*100:.4f}%")

NameError: name 'w1' is not defined

In [ ]:
#Prepare For the test set
X_test = X_test / 255.0

X_test = X_test.reshape(X_test.shape[0], -1)

# convert to tensor
tX_test = torch.tensor(X_test, dtype=torch.float32)
ty_test = torch.tensor(y_test, dtype=torch.long)

print(tX_test.shape)
print(ty_test.shape)

torch.Size([60, 4096])
torch.Size([60])


In [ ]:
#save
np.save("X_test1.npy", X_test)
np.save("y_test1.npy", y_test)

In [ ]:
#Evaluation
Z_test = my_ANN(tX_test, w1, b1, w2, b2, w3, b3)
y_pred_test = torch.argmax(Z_test, dim=1)
accuracy_test = torch.mean((y_pred_test == ty_test).float())
print(f"Accuracy on test set: {accuracy_test*100:.2f}%")

Accuracy on test set: 50.00%
